# L2 logistic regression baseline

Our target is self-reported short sleep, defined as fewer than seven hours. This first run uses `C=1` to establish a validation benchmark. Logistic regression estimates how the ten inputs are associated with the odds of short sleep and gives later models a reference for prediction performance.

## 1. Load the cleaned data and define the outcome

Use the cleaned CSV from the data preparation notebook. Its `enough_sleep` column uses 1 for at least seven hours, so we invert it. The ten predictors are categorical, including columns stored as numeric codes.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

cwd = Path.cwd()
data_dir = next(
    (
        folder / "Data_preprocessing"
        for folder in (cwd, *cwd.parents)
        if (folder / "Data_preprocessing" / "brfss_2022_clean.csv").is_file()
    ),
    None,
)
if data_dir is None:
    raise FileNotFoundError("Open the repository in VS Code so the notebook can find Data_preprocessing.")

# 10 variables that we used in the models, all categorical, and all with no missing values.
features = [
    "age_group", "sex", "income_group", "education_level",
    "employment_status", "marital_status", "children_group",
    "physical_activity", "smoking_status", "state",
]
column_types = dict.fromkeys(features, str)
df = pd.read_csv(data_dir / "brfss_2022_clean.csv", dtype=column_types)
assert df["enough_sleep"].isin([0, 1]).all()
assert df["survey_weight"].gt(0).all()
df["short_sleep"] = 1 - df["enough_sleep"]

print(f"Respondents: {len(df):,}")
print(f"Short-sleep respondents: {df['short_sleep'].sum():,}")

Respondents: 427,573
Short-sleep respondents: 139,673


## 2. Use the existing train, validation, and test assignments

The cleaned CSV includes the shared `split` column. During preprocessing, each state and primary sampling unit (PSU) combination was assigned to one partition. We use those assignments so this baseline and the tree models are compared on the same respondents. Training rows are used for fitting, validation rows are used for comparing settings, and test rows are reserved for final evaluation.

In [ ]:
split_order = ["train", "validation", "test"]

if "split" not in df.columns:
    raise ValueError("No Split column found in the dataset")
assert set(df["split"]) == set(split_order), "Each respondent must have a valid split label."

counts = df["split"].value_counts().reindex(split_order)
display(counts.to_frame("respondents"))

train = df.loc[df["split"].eq("train")]
validation = df.loc[df["split"].eq("validation")]
display(
    df.loc[df["split"].isin(["train", "validation"])]
    .groupby("split")["short_sleep"].mean().mul(100).round(3)
    .to_frame("sample short sleep (%)")
)

,respondents
split,
train,256605
validation,85485
test,85483


,sample short sleep (%)
split,
train,32.623
validation,32.871


## 3. Fit an L2 logistic regression pipeline

One-hot encoding gives each category its own indicator. The pipeline learns the categories from training rows and applies the resulting encoding to validation rows. L2 regularization shrinks large coefficients. Smaller `C` means stronger shrinkage. Training survey weights are divided by their mean so they retain their relative weights while averaging one.

In [3]:
train_weights = train["survey_weight"]
model = make_pipeline(
    OneHotEncoder(handle_unknown="ignore", sparse_output=True),
    LogisticRegression(C=1, solver="lbfgs", max_iter=1_000),
)
model.fit(
    train[features],
    train["short_sleep"],
    logisticregression__sample_weight=train_weights / train_weights.mean(),
)
print(f"Encoded features: {len(model.named_steps['onehotencoder'].get_feature_names_out())}")

Encoded features: 113


## 4. Score on the validation set

Survey-weighted average precision (AP) is our primary metric. It summarizes precision across recall levels when we rank respondents by their predicted probability of short sleep. The constant-score reference has AP equal to the validation set's weighted prevalence. ROC-AUC summarizes how well scores rank short sleepers ahead of other respondents.

In [4]:
val_weights = validation["survey_weight"]
y_val = validation["short_sleep"]
scores = model.predict_proba(validation[features])[:, 1]
weighted_prevalence = np.average(y_val, weights=val_weights)

metrics = pd.DataFrame(
    [
        [weighted_prevalence, 0.5],
        [
            average_precision_score(y_val, scores, sample_weight=val_weights),
            roc_auc_score(y_val, scores, sample_weight=val_weights),
        ],
    ],
    index=["Constant score", "L2 logistic regression"],
    columns=["Weighted AP", "Weighted ROC-AUC"],
)
display(metrics.round(3))

,Weighted AP,Weighted ROC-AUC
Constant score,0.363,0.500
L2 logistic regression,0.464,0.614


## 5. Interpret selected category contrasts

Let $p$ be the model's predicted probability of short sleep (less than seven hours).
Logistic regression models the log-odds of short sleep as a linear combination
of the one-hot encoded predictors:

$$
\log\!\left(\frac{p}{1-p}\right)
= \beta_0 + \sum_{j=1}^{113}\beta_j x_j.
$$

When we switch from a reference category to a comparison category within the
same predictor, the intercept and the contributions of the other predictors
cancel. The change in log-odds is $\beta_{\text{comparison}} -
\beta_{\text{reference}}$. Exponentiating that difference gives the odds ratio:

$$
\mathrm{OR}
= \frac{\mathrm{odds}_{\text{comparison}}}{\mathrm{odds}_{\text{reference}}}
= \exp\!\left(\beta_{\text{comparison}}-\beta_{\text{reference}}\right).
$$

An odds ratio above 1 means higher modeled odds of short sleep for the
comparison category, holding the other predictors fixed. These ratios
describe associations in the fitted model.

In [5]:
encoder = model.named_steps["onehotencoder"]
logistic = model.named_steps["logisticregression"]
coefficients = pd.Series(logistic.coef_[0], index=encoder.get_feature_names_out(features))

contrasts = [
    ("physical_activity", "2", "1", "No leisure-time activity vs. any"),
    ("income_group", "1", "7", "Household income under $10k vs. $50k to under $75k"),
    ("employment_status", "8", "1", "Unable to work vs. employed for wages"),
]
odds_ratios = pd.DataFrame(
    [
        {
            "Contrast": description,
            "Odds ratio": np.exp(coefficients[f"{feature}_{level}"] - coefficients[f"{feature}_{reference}"]),
        }
        for feature, level, reference, description in contrasts
    ]
)
display(odds_ratios.round(3))

,Contrast,Odds ratio
0,No leisure-time activity vs. any,1.235
1,Household income under $10k vs. $50k to under ...,0.992
2,Unable to work vs. employed for wages,1.251


## 6. Tune L2 regularization on the validation set

We randomly sample 20 values of `C` on a log scale from 0.001 to 100. We fit each candidate on the same training set and select the setting with the highest survey-weighted average precision on the validation set. We also report weighted ROC-AUC and compare each candidate with the original `C=1` baseline. The test set is reserved for final evaluation.

In [6]:
from scipy.stats import loguniform
from sklearn.base import clone
from sklearn.model_selection import ParameterSampler

# Search across orders of magnitude. A fixed seed makes the draw reproducible.
c_samples = ParameterSampler(
    {"C": loguniform(1e-3, 100)},
    n_iter=20,
    random_state=42,
)

# Keep the original C=1 result as a reference.
baseline_ap = metrics.loc["L2 logistic regression", "Weighted AP"]
baseline_auc = metrics.loc["L2 logistic regression", "Weighted ROC-AUC"]
results = [
    {"C": 1, "Weighted AP": baseline_ap, "Weighted ROC-AUC": baseline_auc}
]

best_c = 1
best_ap = baseline_ap
best_model = model
normalized_weights = train_weights / train_weights.mean()

for params in c_samples:
    c = params["C"]

    # Start with a fresh copy of the pipeline and change only C.
    candidate = clone(model).set_params(logisticregression__C=c)
    candidate.fit(
        train[features],
        train["short_sleep"],
        logisticregression__sample_weight=normalized_weights,
    )

    # Score every candidate on the same validation rows with the same weights.
    candidate_scores = candidate.predict_proba(validation[features])[:, 1]
    ap = average_precision_score(y_val, candidate_scores, sample_weight=val_weights)
    auc = roc_auc_score(y_val, candidate_scores, sample_weight=val_weights)
    results.append({"C": c, "Weighted AP": ap, "Weighted ROC-AUC": auc})

    if ap > best_ap:
        best_c = c
        best_ap = ap
        best_model = candidate

# Rank by the full-precision AP and round only for display.
tuning_results = pd.DataFrame(results).sort_values(
    "Weighted AP", ascending=False, kind="stable"
).reset_index(drop=True)

tuning_results["AP gain (percentage points)"] = 100 * (
    tuning_results["Weighted AP"] - baseline_ap
)

display(tuning_results.round(3))
print(f"Best C: {best_c:.3f}")
print(f"Best validation weighted AP: {best_ap:.3f}")
print(f"AP gain vs. C=1: {100 * (best_ap - baseline_ap):.3f} percentage points")

,C,Weighted AP,Weighted ROC-AUC,AP gain (percentage points)
0,0.012,0.464,0.614,0.009
1,0.008,0.464,0.614,0.008
2,0.008,0.464,0.614,0.008
3,0.006,0.464,0.614,0.007
4,0.006,0.464,0.614,0.007
5,0.029,0.464,0.614,0.005
6,0.033,0.464,0.614,0.005
7,0.002,0.464,0.614,0.003
8,0.075,0.464,0.614,0.002
9,0.144,0.464,0.614,0.001


Best C: 0.012
Best validation weighted AP: 0.464
AP gain vs. C=1: 0.009 percentage points
